# Thermal Conditions, Illumination, and Extremes Preprocessing: Earth, Moon, and Mars

A single notebook handles all three celestial bodies using **consistent column definitions and the same H3 resolution (5)**.

The notebook can be executed with `python modules_pretraitement/thermal_light_extrems/scripts/lancer_3_corps.py`, or opened directly in VS Code using the repository's Python kernel.

Generated results are stored in `pretreatment_data/thermal_light_extrems/` at the repository root.

```text
pretreatment_data/thermal_light_extrems/
├── thermal_earth_h3_5.parquet
├── illumination_earth_h3_5.parquet
├── extremes_earth_h3_5.parquet
├── earth.duckdb
├── thermal_moon_h3_5.parquet
├── illumination_moon_h3_5.parquet
├── extremes_moon_h3_5.parquet
├── moon.duckdb
├── thermal_mars_h3_5.parquet
├── illumination_mars_h3_5.parquet
├── extremes_mars_h3_5.parquet
└── mars.duckdb
```

## Data Sources

| | Earth | Moon | Mars |
|---|---|---|---|
| Surface temperature (`lst_*`) | MODIS **MOD11C3** + **MYD11C3** v061 (Terra + Aqua), monthly, 0.05° | **Diviner** (LRO), Global Cumulative Product, 0.5° × local time | **Mars Climate Database** v6.2, `tsurf` |
| Air temperature | MERRA-2 **M2SMNXSLV** (`T2MMAX`, `T2MMEAN`, `T2MMIN`) | Not available (`NaN`) | MCD, lowest atmospheric level of `temp` |
| Solar radiation, infrared radiation, clouds | MERRA-2 **M2TMNXRAD** (`SWGDN`, `LWGAB`, `CLDTOT`) | Calculated; 0; 0 | MCD `fluxsurf_dn_sw`, `fluxsurf_lw`; clouds: `NaN` |
| Fraction of time exposed to sunlight | Astronomical calculation (latitude, axial tilt) | Astronomical calculation; **LOLA** illumination map at the poles | Astronomical calculation |

### Data Requirements

**Earth:** A NASA Earthdata account configured in `~/.netrc` is sufficient to access MODIS and MERRA-2 datasets. For MERRA-2, the **NASA GESDISC DATA ARCHIVE** application must also be authorized in the Earthdata user profile.

**Moon:** No manual preparation is required. PDS files are downloaded automatically.

**Mars:** MCD files must be manually copied into `inputs/mcd/`, which is shared with the `wind_erosion_atmosphere` module. See the Mars setup step for details.

## Common Definitions

- **Observation:** One source pixel × one date.
  - **Earth:** One month × one satellite.
  - **Moon:** One local time of the lunar cycle.
  - **Mars:** One Martian month × one local time.

  The `mean`, `min`, `max`, `p05`, and `p95` statistics are calculated using all valid observations within each H3 cell.

- **Daytime / Nighttime:** Defined using the local solar times of MODIS satellite overpasses:
  - **Daytime:** 10:30 AM and 1:30 PM.
  - **Nighttime:** 10:30 PM and 1:30 AM.

  These are the only available observation times for Earth. The same times are applied to the Moon and Mars to ensure consistency.

- **Hot / Cold Extremes:** Daytime observations are classified using fixed temperature thresholds:
  - **Hot:** ≥ 313 K.
  - **Cold:** ≤ 253 K.

  The corresponding fractions are calculated relative to the total number of valid daytime observations.

- **Units:**
  - Temperature: Kelvin (K).
  - Radiation: W/m².
  - Duration: Hours per celestial-body day:
    - Earth: 24 hours.
    - Mars: 24.6597 hours.
    - Moon: 708.7 hours.
  - Fractions: Values between 0 and 1.

- **H3 Cell Assignment:** Each H3 cell receives the source pixels whose centers fall within its boundaries. If no source pixel center falls inside a cell (when the source resolution is coarser than the H3 cell), the source pixel containing the H3 cell's center is assigned instead.

- **Masked Pixel Count:** `masked_pixel_count = 0`, since no category-based masking is applied in this module.

- **Coverage:** `coverage = valid observations / possible observations`.

- **Reliability:** `reliability = coverage`. This is a **temporary definition**, applied consistently across all three celestial bodies.

In [ ]:
import os, glob, re, time, math, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import h3
warnings.filterwarnings("ignore", category=RuntimeWarning)   # cellules sans mesure -> NaN, voulu

# ------------------------------------------------------------------ paramètres
FAIRE = {"moon": True, "earth": True, "mars": True}
RESOLUTION = 5
ANNEE_TERRE = 2023
ZONE_TERRE = None            # None = toute la Terre ; (ouest, sud, est, nord) : calcul limité à cette boîte
# Extrêmes relatifs à chaque corps : une observation de jour est « chaude » au-dessus du 90e centile des
# températures de jour du corps, « froide » sous le 10e (Mars n'atteint jamais 313 K, la Lune y est toujours).
CENTILES_EXTREMES = (10, 90)
HEURES = {"jour": [10.5, 13.5], "nuit": [22.5, 1.5]}    # passages MODIS (Terra, Aqua), heure solaire locale
CORPS = {
    "earth": {"cycle_h": 24.0, "obliquite": 23.44, "rayon_km": 6371.0},
    "moon":  {"cycle_h": 708.7, "obliquite": 1.54, "rayon_km": 1737.4},
    "mars":  {"cycle_h": 24.6597, "obliquite": 25.19, "rayon_km": 3389.5},
}
# Taille du pixel de chaque source (km), pour la fiabilité : une valeur recopiée depuis une grille plus
# grossière que la cellule vaut moins. fiabilité = couverture / (1 + log2(pixel source / taille de cellule)).
SOURCE_KM = {
    "earth": {"thermal": 5.6, "extremes": 5.6, "illumination": 62.0},     # MODIS 0,05° ; MERRA-2 0,5° × 0,625°
    "moon":  {"thermal": 15.2, "extremes": 15.2, "illumination": 0.24},   # Diviner 0,5° ; calcul exact, LOLA 240 m aux pôles
    "mars":  {"thermal": 272.0, "extremes": 272.0, "illumination": 272.0},  # MCD 5,625° × 3,75°
}
# Racine du dépôt : premier dossier parent qui contient requirements.txt et modules_pretraitement/.
RACINE = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "requirements.txt").is_file() and (p / "modules_pretraitement").is_dir()), Path.cwd())
DOSSIER = str(RACINE / "cache" / "thermal_light_extrems")                  # téléchargements temporaires, effacés après lecture
SORTIE = str(RACINE / "pretreatment_data" / "thermal_light_extrems")       # résultats
DOSSIER_MCD = str(RACINE / "inputs" / "mcd")                              # Mars Climate Database, copiée à la main
os.makedirs(DOSSIER, exist_ok=True); os.makedirs(SORTIE, exist_ok=True)
print("Résultats dans", SORTIE)

def telecharger_fichier(url, chemin, essais=3):
    # Écrit dans chemin.part et ne renomme qu'une fois la taille annoncée par le serveur atteinte :
    # un fichier coupé en route n'est jamais lu (il manquait une partie de la Lune, 80 à 70° S).
    import urllib.request
    partiel = chemin + ".part"
    for essai in range(essais):
        try:
            with urllib.request.urlopen(url, timeout=120) as r, open(partiel, "wb") as f:
                attendu = int(r.headers.get("Content-Length") or 0)
                shutil.copyfileobj(r, f, length=1 << 20)
            recu = os.path.getsize(partiel)
            if attendu and recu != attendu:
                raise IOError(f"fichier incomplet : {recu:,} octets sur {attendu:,}")
            os.replace(partiel, chemin)
            return
        except Exception as e:
            print(f"  échec du téléchargement de {url} (essai {essai + 1}/{essais}) : {e}")
            if os.path.exists(partiel):
                os.remove(partiel)
            time.sleep(5 * (essai + 1))

COLONNES = {
    "thermal": ["lst_day_mean", "lst_day_min", "lst_day_max", "lst_night_mean", "lst_night_min", "lst_night_max",
                "air_temperature_mean", "air_temperature_min", "air_temperature_max",
                "pixel_count", "valid_pixel_count", "masked_pixel_count", "coverage", "source_resolution_km", "reliability"],
    "illumination": ["shortwave_radiation_mean", "longwave_radiation_mean", "sunshine_duration_mean",
                     "daylight_duration_mean", "cloud_fraction_mean", "daylight_fraction",
                     "daylight_fraction_min", "daylight_fraction_max",
                     "pixel_count", "valid_pixel_count", "masked_pixel_count", "coverage", "source_resolution_km", "reliability"],
    "extremes": ["lst_day_p05", "lst_day_p95", "lst_night_p05", "lst_night_p95", "air_temperature_p05", "air_temperature_p95",
                 "hot_pixel_count", "cold_pixel_count", "hot_fraction", "cold_fraction",
                 "pixel_count", "valid_pixel_count", "masked_pixel_count", "coverage", "source_resolution_km", "reliability"],
}
RESULTATS = {}
SEUILS = {}

## Shared Utilities

The same functions are used across all three celestial bodies: mapping source pixels to H3 cells, computing observation statistics, calculating the fraction of time exposed to sunlight, and generating and exporting the three output tables.

In [ ]:
def toutes_les_cellules(res=RESOLUTION):
    cellules = np.array([c for r0 in h3.get_res0_cells() for c in h3.cell_to_children(r0, res)])
    centres = np.array([h3.cell_to_latlng(c) for c in cellules])
    return cellules, centres[:, 0], centres[:, 1]

def vers_cellules(lats, lngs, res=RESOLUTION):
    return np.array([h3.latlng_to_cell(float(a), float(b), res) for a, b in zip(lats, lngs)])

def indices_grille(valeurs, premier, pas, n, periodique=False):
    # Indice de la case d'une grille régulière la plus proche (pas négatif accepté).
    i = np.rint((np.asarray(valeurs) - premier) / pas).astype(int)
    return np.mod(i, n) if periodique else np.clip(i, 0, n - 1)

def stats_lignes(X):
    # Statistiques de chaque ligne (un pixel source, ses observations en colonnes).
    return {"mean": np.nanmean(X, axis=1), "min": np.nanmin(X, axis=1), "max": np.nanmax(X, axis=1),
            "p05": np.nanpercentile(X, 5, axis=1), "p95": np.nanpercentile(X, 95, axis=1),
            "n_valides": np.isfinite(X).sum(axis=1), "n_total": np.full(len(X), X.shape[1])}

def agreger(n, cell_pix, idx_centre, series, comptes=None):
    # Statistiques par cellule.
    #   n          : nombre de cellules de sortie
    #   cell_pix   : (n_pix,) indice de la cellule qui contient le centre de chaque pixel source (-1 : aucune)
    #   idx_centre : (n,) pixel source qui contient le centre de chaque cellule (-1 : aucun)
    #   series     : {nom: tableau (n_pix, k)} observations, NaN = manquante
    #   comptes    : {nom: (serie, fonction)} nombre d'observations qui vérifient la fonction
    comptes = comptes or {}
    nb = np.bincount(cell_pix[cell_pix >= 0], minlength=n)
    # Cellules à 0 ou 1 pixel : on lit directement un pixel (le sien, ou celui qui contient son centre)
    pix = np.full(n, -1)
    pv = np.where(cell_pix >= 0)[0]
    seuls = pv[nb[cell_pix[pv]] == 1]
    pix[cell_pix[seuls]] = seuls
    vides = nb == 0
    pix[vides] = idx_centre[vides]
    utiles = np.unique(pix[pix >= 0])
    pos = np.searchsorted(utiles, pix)
    ok = pix >= 0
    sortie = {}
    for nom, X in series.items():
        s = stats_lignes(X[utiles])
        sortie[nom] = {k: np.full(n, np.nan if k not in ("n_valides", "n_total") else 0, dtype=np.float64) for k in s}
        for k, v in s.items():
            sortie[nom][k][ok] = v[pos[ok]]
    for nom, (serie, f) in comptes.items():
        c = f(series[serie][utiles]).sum(axis=1)
        sortie[nom] = np.zeros(n); sortie[nom][ok] = c[pos[ok]]
    # Cellules à plusieurs pixels : toutes leurs observations ensemble
    multi = np.where(nb >= 2)[0]
    if len(multi):
        ordre = pv[np.argsort(cell_pix[pv], kind="stable")]
        cles = cell_pix[ordre]
        debut, fin = np.searchsorted(cles, multi), np.searchsorted(cles, multi, side="right")
        for cellule, a, b in zip(multi, debut, fin):
            lignes = ordre[a:b]
            for nom, X in series.items():
                x = X[lignes].ravel()
                f = x[np.isfinite(x)]
                d = sortie[nom]
                d["n_total"][cellule] = x.size; d["n_valides"][cellule] = f.size
                if f.size:
                    d["mean"][cellule], d["min"][cellule], d["max"][cellule] = f.mean(), f.min(), f.max()
                    d["p05"][cellule], d["p95"][cellule] = np.percentile(f, [5, 95])
            for nom, (serie, fn) in comptes.items():
                sortie[nom][cellule] = fn(series[serie][lignes]).sum()
    return sortie

def fractions_jour(lats, obliquite_deg):
    # Part du cycle (jour solaire) où le Soleil est au-dessus de l'horizon, sans relief ni réfraction :
    # moyenne sur l'année (0,5 presque partout), jour le plus court et jour le plus long de l'année
    # (0 = nuit polaire, 1 = soleil de minuit). Ce sont ces deux derniers qui distinguent les latitudes.
    grille = np.round(np.asarray(lats), 1)
    uniques, inverse = np.unique(grille, return_inverse=True)
    ls = np.linspace(0, 2 * np.pi, 360, endpoint=False)
    decl = np.arcsin(np.sin(np.radians(obliquite_deg)) * np.sin(ls))
    cos_h = np.clip(-np.tan(np.radians(uniques))[:, None] * np.tan(decl)[None, :], -1, 1)
    f = np.arccos(cos_h) / np.pi
    return tuple(x[inverse].astype(np.float32) for x in (f.mean(axis=1), f.min(axis=1), f.max(axis=1)))

def seuils_extremes(X_jour, lats, corps, n_tirages=200_000):
    # Seuils froid / chaud du corps : 10e et 90e centiles des températures de jour, sur des pixels tirés
    # au hasard en proportion de leur surface (cos latitude), pour ne pas surreprésenter les pôles.
    rng = np.random.default_rng(0)
    p = np.cos(np.radians(np.asarray(lats, dtype=float))).clip(1e-6)
    lignes = rng.choice(len(p), size=min(n_tirages, len(p)), p=p / p.sum())
    x = X_jour[lignes].ravel()
    froid, chaud = np.nanpercentile(x[np.isfinite(x)], CENTILES_EXTREMES)
    print(f"{corps} : extrêmes relatifs, froid <= {froid:.1f} K, chaud >= {chaud:.1f} K "
          f"({CENTILES_EXTREMES[0]}e et {CENTILES_EXTREMES[1]}e centiles des températures de jour)")
    SEUILS[corps] = {"froid_K": round(float(froid), 2), "chaud_K": round(float(chaud), 2)}
    return {"hot": ("jour", lambda v: v >= chaud), "cold": ("jour", lambda v: v <= froid)}

def facteur_resolution(corps, table):
    # 1 si la source est au moins aussi fine que la cellule ; 1/2 si elle est 2 fois plus grossière ; 1/3 à 4 fois…
    aire = h3.average_hexagon_area(RESOLUTION, "km^2") * (CORPS[corps]["rayon_km"] / 6371.0) ** 2
    rapport = max(1.0, SOURCE_KM[corps][table] / math.sqrt(aire))
    return 1.0 / (1.0 + math.log2(rapport))

def construire_tables(corps, cellules, lst, air, illum):
    # lst : sortie d'agreger avec "jour", "nuit", "hot", "cold" ; air : idem avec "moy", "min", "max" (ou None)
    # illum : dict de colonnes d'illumination + "n_total", "n_valides", "fiabilite"
    n = len(cellules)
    base = pd.DataFrame({"body": corps, "resolution": RESOLUTION, "h3": cellules})
    f32 = lambda a: np.asarray(a, dtype=np.float32)
    nan = np.full(n, np.nan, dtype=np.float32)
    j, nu = lst["jour"], lst["nuit"]
    total = (j["n_total"] + nu["n_total"]).astype(np.int64)
    valides = (j["n_valides"] + nu["n_valides"]).astype(np.int64)
    couverture = f32(np.where(total > 0, valides / np.maximum(total, 1), 0))
    comptes = lambda table: {"pixel_count": total, "valid_pixel_count": valides, "masked_pixel_count": np.zeros(n, dtype=np.int64),
                             "coverage": couverture, "source_resolution_km": np.float32(SOURCE_KM[corps][table]),
                             "reliability": f32(couverture * facteur_resolution(corps, table))}
    a = air or {}
    thermal = base.assign(
        lst_day_mean=f32(j["mean"]), lst_day_min=f32(j["min"]), lst_day_max=f32(j["max"]),
        lst_night_mean=f32(nu["mean"]), lst_night_min=f32(nu["min"]), lst_night_max=f32(nu["max"]),
        air_temperature_mean=f32(a["moy"]["mean"]) if a else nan,
        air_temperature_min=f32(a["min"]["min"]) if a else nan,
        air_temperature_max=f32(a["max"]["max"]) if a else nan, **comptes("thermal"))
    jour_valides = j["n_valides"]
    frac = lambda c: f32(np.where(jour_valides > 0, c / np.maximum(jour_valides, 1), np.nan))
    extremes = base.assign(
        lst_day_p05=f32(j["p05"]), lst_day_p95=f32(j["p95"]), lst_night_p05=f32(nu["p05"]), lst_night_p95=f32(nu["p95"]),
        air_temperature_p05=f32(a["moy"]["p05"]) if a else nan, air_temperature_p95=f32(a["moy"]["p95"]) if a else nan,
        hot_pixel_count=lst["hot"].astype(np.int64), cold_pixel_count=lst["cold"].astype(np.int64),
        hot_fraction=frac(lst["hot"]), cold_fraction=frac(lst["cold"]), **comptes("extremes"))
    it, iv = illum["n_total"].astype(np.int64), illum["n_valides"].astype(np.int64)
    ic = f32(np.where(it > 0, iv / np.maximum(it, 1), 0))
    illumination = base.assign(
        shortwave_radiation_mean=f32(illum["shortwave"]), longwave_radiation_mean=f32(illum["longwave"]),
        sunshine_duration_mean=f32(illum["sunshine"]), daylight_duration_mean=f32(illum["daylight_h"]),
        cloud_fraction_mean=f32(illum["cloud"]), daylight_fraction=f32(illum["daylight_fraction"]),
        daylight_fraction_min=f32(illum["daylight_min"]), daylight_fraction_max=f32(illum["daylight_max"]),
        pixel_count=it, valid_pixel_count=iv, masked_pixel_count=np.zeros(n, dtype=np.int64),
        coverage=ic, source_resolution_km=np.float32(SOURCE_KM[corps]["illumination"]),
        reliability=f32(illum.get("fiabilite", ic) * facteur_resolution(corps, "illumination")))
    tables = {"thermal": thermal, "illumination": illumination, "extremes": extremes}
    for nom, t in tables.items():
        tables[nom] = t[["body", "resolution", "h3", *COLONNES[nom]]]
    return tables

def ecrire(corps, tables):
    import sys
    sys.path.insert(0, str(RACINE / "scripts"))
    from commun import reconstruire_duckdb
    for nom, t in tables.items():
        t.to_parquet(f"{SORTIE}/{nom}_{corps}_h3_{RESOLUTION}.parquet", index=False, compression="zstd")
    # {corps}.duckdb : copie des Parquet, refaite entièrement (évite l'erreur de version de DuckDB
    # quand le dossier a été copié depuis une autre machine)
    reconstruire_duckdb(Path(SORTIE), corps)
    if corps in SEUILS:
        import json
        with open(f"{SORTIE}/seuils_extremes_{corps}.json", "w") as f:
            json.dump(SEUILS[corps], f)
    RESULTATS[corps] = tables
    t = tables["thermal"]
    print(f"{corps} : {len(t):,} cellules écrites ; couverture moyenne {t.coverage.mean():.1%} ; "
          f"lst_day_mean médiane {t.lst_day_mean.median():.0f} K, lst_night_mean médiane {t.lst_night_mean.median():.0f} K")

def controle(corps, sites):
    t = RESULTATS[corps]
    th, il, ex = (t[k].set_index("h3") for k in ("thermal", "illumination", "extremes"))
    lignes = []
    for nom, (la, ln) in sites.items():
        c = h3.latlng_to_cell(la, ln, RESOLUTION)
        if c not in th.index:
            lignes.append({"site": nom, "remarque": "cellule absente (hors données)"}); continue
        lignes.append({"site": nom, "jour moy (K)": th.loc[c, "lst_day_mean"], "nuit moy (K)": th.loc[c, "lst_night_mean"],
                       "air moy (K)": th.loc[c, "air_temperature_mean"], "solaire (W/m²)": il.loc[c, "shortwave_radiation_mean"],
                       "jour court": il.loc[c, "daylight_fraction_min"], "jour long": il.loc[c, "daylight_fraction_max"],
                       "hot": ex.loc[c, "hot_fraction"],
                       "cold": ex.loc[c, "cold_fraction"], "couverture": th.loc[c, "coverage"]})
    return pd.DataFrame(lignes).round(2)

CELLULES, LAT_C, LNG_C = toutes_les_cellules()
print(f"{len(CELLULES):,} cellules H3 de résolution {RESOLUTION}")

## Moon

**Diviner (LRO), Global Cumulative Product:** 18 fixed-width text files (one per 10° latitude band, ~156 MB each, ~2.8 GB in total), downloaded, read, and deleted sequentially.

**File format (PDS4 label):**
- Record length: 113 bytes.
- Longitude: bytes 1–7.
- Latitude: bytes 10–15.
- Local time: bytes 18–23.
- `tbol`: bytes 103–111.
- `-9999`: Indicates missing measurements.

**LOLA, Mean Solar Visibility** (from 65° latitude to the pole, 240 m/pixel): Raw image containing 6420 × 6420 16-bit integers, with values calculated as `DN × 0.00004`, using a polar stereographic projection.

The image orientation is verified through correlation with daytime surface temperatures.

In [ ]:
if FAIRE["moon"]:
    URL_DIVINER = "https://pds-geosciences.wustl.edu/lro/urn-nasa-pds-lro_diviner_derived1/data_derived_gcp/"
    BANDES = ["90s80s", "80s70s", "70s60s", "60s50s", "50s40s", "40s30s", "30s20s", "20s10s", "10s00s",
              "00n10n", "10n20n", "20n30n", "30n40n", "40n50n", "50n60n", "60n70n", "70n80n", "80n90n"]
    LIGNE, ENTETE = 113, 113
    CHAMPS = {"lon": (0, 7), "lat": (9, 15), "heure": (17, 23), "tbol": (102, 111)}

    def lire_bande(chemin):
        brut = open(chemin, "rb").read()
        corps = np.frombuffer(brut, dtype=np.uint8, offset=ENTETE)
        n = corps.size // LIGNE
        lignes = corps[: n * LIGNE].reshape(n, LIGNE)
        return {k: np.ascontiguousarray(lignes[:, a:b]).view(f"S{b - a}").ravel().astype(np.float64) for k, (a, b) in CHAMPS.items()}

    T_LUNE = np.full((360, 720, 96), np.nan, dtype=np.float32)    # latitude x longitude x quart d'heure local
    t0 = time.time()
    for bande in BANDES:
        nom = f"global_cumul_avg_cyl_{bande}_002.tab"
        chemin = f"{DOSSIER}/{nom}"
        if not os.path.exists(chemin):
            telecharger_fichier(URL_DIVINER + nom, chemin)
        if not os.path.exists(chemin) or os.path.getsize(chemin) < 1000:
            print(f"{bande} : téléchargement impossible, bande ignorée"); continue
        d = lire_bande(chemin)
        # Contrôle : une bande complète a des mesures dans presque toutes ses cases de 0,5° (lat x lon)
        cases = np.unique(np.floor(d["lat"] * 2) * 1000 + np.floor(((d["lon"] + 180) % 360) * 2)).size
        if cases < 0.9 * 20 * 720:
            print(f"{bande} : seulement {cases / (20 * 720):.0%} des cases présentes, fichier probablement tronqué ; nouveau téléchargement")
            os.replace(chemin, chemin + ".incomplet")
            telecharger_fichier(URL_DIVINER + nom, chemin)
            if os.path.exists(chemin):
                d = lire_bande(chemin)
                os.remove(chemin + ".incomplet")
            else:                                  # pas mieux : on garde ce qu'on a lu, l'audit montrera le trou
                print(f"{bande} : nouveau téléchargement impossible, bande gardée telle quelle")
                os.replace(chemin + ".incomplet", chemin)
        t = np.where(d["tbol"] <= -9998, np.nan, d["tbol"])
        aberrant = t > 405                     # au-dessus du maximum physique (~400 K au point subsolaire)
        if aberrant.any():
            print(f"{bande} : {int(aberrant.sum())} mesures au-dessus de 405 K écartées")
            t[aberrant] = np.nan
        il = np.floor((d["lat"] + 90) * 2).astype(int).clip(0, 359)
        jl = np.floor(((d["lon"] + 180) % 360) * 2).astype(int).clip(0, 719)
        hl = np.floor(d["heure"] * 4).astype(int).clip(0, 95)
        T_LUNE[il, jl, hl] = t
        print(f"{bande} : {np.isfinite(t).mean():.0%} avec mesure ({time.time() - t0:.0f} s)")
        os.remove(chemin)

    # Carte LOLA de visibilité du Soleil aux deux pôles
    URL_LOLA = "https://pds-geosciences.wustl.edu/lro/lro-l-lola-3-rdr-v1/lrolol_1xxx/extras/illumination/img/avgvisib_65{p}_240m_201608.img"
    N_LOLA = 6420
    LOLA = {}
    for p in ["s", "n"]:
        chemin = f"{DOSSIER}/avgvisib_65{p}.img"
        if not os.path.exists(chemin) or os.path.getsize(chemin) != N_LOLA ** 2 * 2:
            telecharger_fichier(URL_LOLA.format(p=p), chemin)
        if os.path.exists(chemin) and os.path.getsize(chemin) == N_LOLA ** 2 * 2:
            LOLA[p] = np.fromfile(chemin, dtype="<i2").reshape(N_LOLA, N_LOLA).astype(np.float32) * 4e-5
            print(f"LOLA 65{p.upper()} : visibilité {LOLA[p].min():.2f} à {LOLA[p].max():.2f}")
        else:
            print(f"LOLA 65{p.upper()} : téléchargement impossible, valeur géométrique gardée")

In [ ]:
def visibilite_lola(lats, lngs, orientation=(1, 1)):
    sx, sy = orientation
    sortie = np.full(len(lats), np.nan)
    for p, carte in LOLA.items():
        m = (lats <= -65) if p == "s" else (lats >= 65)
        if not m.any():
            continue
        la, ln = np.radians(lats[m]), np.radians(lngs[m])
        rho = 2 * 1737400.0 * np.tan(np.pi / 4 + (la if p == "s" else -la) / 2)
        x, y = rho * np.sin(ln), rho * np.cos(ln) * (1 if p == "s" else -1)
        col = np.rint(3209.5 + sx * x / 240.0).astype(int)
        lig = np.rint(3209.5 - sy * y / 240.0).astype(int)
        dedans = (col >= 0) & (col < N_LOLA) & (lig >= 0) & (lig < N_LOLA)
        v = np.full(m.sum(), np.nan)
        v[dedans] = carte[lig[dedans], col[dedans]]
        sortie[m] = np.clip(v, 0, 1)
    return sortie

if FAIRE["moon"]:
    h_lune = (np.arange(96) + 0.5) / 4
    # Quarts d'heure locaux entre les deux passages MODIS de jour (10 h 30 - 13 h 30) et de nuit (22 h 30 - 1 h 30)
    q_jour = (h_lune >= 10.5) & (h_lune < 13.5)
    q_nuit = (h_lune >= 22.5) | (h_lune < 1.5)
    pix_lat = (-89.75 + 0.5 * np.arange(360)).repeat(720)
    pix_lng = np.tile(-179.75 + 0.5 * np.arange(720), 360)
    X = T_LUNE.reshape(-1, 96)
    series = {"jour": X[:, q_jour], "nuit": X[:, q_nuit]}

    index = pd.Index(CELLULES)
    cell_pix = index.get_indexer(vers_cellules(pix_lat, pix_lng))
    idx_centre = (np.floor((LAT_C + 90) * 2).astype(int).clip(0, 359) * 720 + np.floor((LNG_C + 180) * 2).astype(int).clip(0, 719))
    lst = agreger(len(CELLULES), cell_pix, idx_centre, series, seuils_extremes(series["jour"], pix_lat, "moon"))

    # Illumination
    fraction, f_min, f_max = fractions_jour(LAT_C, CORPS["moon"]["obliquite"])
    geometrique = fraction.copy()
    fiab = np.ones(len(CELLULES), dtype=np.float32)
    pole = np.abs(LAT_C) >= 65
    if LOLA:
        test = np.abs(LAT_C) >= 80
        scores = {}
        for o in [(1, 1), (-1, 1), (1, -1), (-1, -1)]:
            v = visibilite_lola(LAT_C[test], LNG_C[test], o); t = lst["jour"]["mean"][test]
            ok = np.isfinite(v) & np.isfinite(t)
            scores[o] = np.corrcoef(v[ok], t[ok])[0, 1] if ok.sum() > 100 else -1
        orientation = max(scores, key=scores.get)
        print("Corrélation visibilité LOLA / température de jour :", {k: round(float(v), 3) for k, v in scores.items()}, "-> retenue", orientation)
        if scores[orientation] < 0.2:
            print("ATTENTION : corrélation faible, orientation LOLA incertaine.")
        vis = visibilite_lola(LAT_C[pole], LNG_C[pole], orientation)
        fraction[pole] = np.where(np.isfinite(vis), vis, fraction[pole])
        fiab[np.where(pole)[0][~np.isfinite(vis)]] = 0.3
        # LOLA ne donne que la moyenne : jour le plus court et le plus long mis à la même échelle (approximation)
        rapport = np.where(geometrique[pole] > 0, fraction[pole] / np.maximum(geometrique[pole], 1e-6), 0)
        f_min[pole], f_max[pole] = np.clip(f_min[pole] * rapport, 0, 1), np.clip(f_max[pole] * rapport, 0, 1)
    else:
        fiab[pole] = 0.3
    ondes_courtes = 1361.0 / math.pi * np.cos(np.radians(LAT_C))
    ondes_courtes = np.where(pole, ondes_courtes * fraction / 0.5, ondes_courtes)
    n = len(CELLULES)
    illum = {"shortwave": ondes_courtes, "longwave": np.zeros(n), "cloud": np.zeros(n),
             "daylight_fraction": fraction, "daylight_min": f_min, "daylight_max": f_max,
             "daylight_h": fraction * CORPS["moon"]["cycle_h"],
             "sunshine": fraction * CORPS["moon"]["cycle_h"], "n_total": np.ones(n), "n_valides": np.ones(n), "fiabilite": fiab}
    ecrire("moon", construire_tables("moon", CELLULES, lst, None, illum))
    del T_LUNE, X, series
    display(controle("moon", {"Apollo 11": (0.674, 23.473), "Apollo 15": (26.132, 3.634), "Tycho": (-43.31, -11.36),
                              "Chang'e 4": (-45.444, 177.599), "Shackleton (fond)": (-89.67, 129.78)}))

---

## Earth

- **MOD11C3 and MYD11C3 (v061):** Monthly daytime and nighttime land surface temperatures on a global 0.05° grid (7200 × 3600), obtained from the Terra and Aqua satellites. A total of 24 files per year are downloaded, processed, and deleted sequentially. Temperature values are calculated as `DN × 0.02 K`. Valid measurements are available only over land.

- **MERRA-2 M2SMNXSLV:** Air temperature at 2 meters above the surface, including monthly averages of daily maximum, mean, and minimum temperatures, at a spatial resolution of 0.5° × 0.625°.

- **MERRA-2 M2TMNXRAD:** Downward shortwave radiation at the surface (`SWGDN`), longwave radiation absorbed by the surface (`LWGAB`, approximately equivalent to downward infrared radiation), and total cloud fraction (`CLDTOT`).

**Mean sunshine duration** (`sunshine_duration_mean`) is estimated using the following formula:

```text
sunshine_duration_mean = daylight_duration × (1 − cloud_fraction)
```

In [ ]:
if FAIRE["earth"]:
    import earthaccess
    # Cherche d'abord EARTHDATA_USERNAME / EARTHDATA_PASSWORD, puis ~/.netrc, et ne demande
    # l'identifiant qu'en dernier recours : un lancement sans écran (nbconvert, ssh) ne bloque pas.
    if not earthaccess.login().authenticated:
        raise RuntimeError("Connexion Earthdata impossible : voir modules_pretraitement/thermal_light_extrems/README.md")
    periode = (f"{ANNEE_TERRE}-01-01", f"{ANNEE_TERRE}-12-31")

    def recuperer(g, dossier):
        # show_progress n'existe pas dans les anciennes versions d'earthaccess (Python 3.9)
        try:
            return earthaccess.download([g], local_path=dossier, show_progress=False)
        except TypeError:
            return earthaccess.download([g], local_path=dossier)

    def telecharger(nom_court, version=None):
        kw = {"short_name": nom_court, "temporal": periode}
        if version: kw["version"] = version
        if ZONE_TERRE: kw["bounding_box"] = ZONE_TERRE
        g = earthaccess.search_data(**kw)
        print(f"{nom_court} : {len(g)} fichiers")
        return g

    # --- MODIS CMG mensuel (Terra et Aqua)
    from pyhdf.SD import SD, SDC
    def lire_lst(chemin):
        sd = SD(chemin, SDC.READ)
        sortie = {}
        for cle, debut in [("jour", "LST_Day"), ("nuit", "LST_Night")]:
            nom = [k for k in sd.datasets() if k.startswith(debut)][0]
            s = sd.select(nom); a = s.attributes()
            dn = s[:].astype(np.uint16)
            bas, haut = a.get("valid_range", [1, 65535])
            dn[(dn < bas) | (dn > haut) | (dn == a.get("_FillValue", 0))] = 0
            sortie[cle] = dn
            echelle = float(a.get("scale_factor", 0.02))
        sd.end()
        return sortie, echelle

    granules = telecharger("MOD11C3", "061") + telecharger("MYD11C3", "061")
    DN = {"jour": np.zeros((len(granules), 3600, 7200), np.uint16), "nuit": np.zeros((len(granules), 3600, 7200), np.uint16)}
    echelle_lst = 0.02
    for k, g in enumerate(granules):
        chemin = recuperer(g, f"{DOSSIER}/modis")[0]
        lu, echelle_lst = lire_lst(str(chemin))
        DN["jour"][k], DN["nuit"][k] = lu["jour"], lu["nuit"]
        os.remove(chemin)
        print(f"  {k + 1}/{len(granules)} {os.path.basename(str(chemin))}")

    terre = (DN["jour"] > 0).any(axis=0) | (DN["nuit"] > 0).any(axis=0)
    lig, col = np.nonzero(terre)
    pix_lat, pix_lng = 89.975 - 0.05 * lig, -179.975 + 0.05 * col
    if ZONE_TERRE:
        w, s_, e, nn = ZONE_TERRE
        garde = (pix_lng >= w) & (pix_lng <= e) & (pix_lat >= s_) & (pix_lat <= nn)
        lig, col, pix_lat, pix_lng = lig[garde], col[garde], pix_lat[garde], pix_lng[garde]
    def en_kelvins(dn):
        v = dn.astype(np.float32) * echelle_lst
        v[dn == 0] = np.nan
        return v
    series_t = {"jour": en_kelvins(DN["jour"][:, lig, col].T), "nuit": en_kelvins(DN["nuit"][:, lig, col].T)}
    del DN, terre
    print(f"{len(pix_lat):,} pixels de terre, {series_t['jour'].shape[1]} observations de jour et de nuit chacun")

In [ ]:
if FAIRE["earth"]:
    t0 = time.time()
    cell_pix_txt = vers_cellules(pix_lat, pix_lng)
    CELLULES_T, cell_pix = np.unique(cell_pix_txt, return_inverse=True)
    centres = np.array([h3.cell_to_latlng(c) for c in CELLULES_T])
    LAT_T, LNG_T = centres[:, 0], centres[:, 1]
    idx_centre = np.full(len(CELLULES_T), -1)   # chaque cellule de terre contient au moins un pixel
    print(f"{len(CELLULES_T):,} cellules de terre ({time.time() - t0:.0f} s)")
    lst_t = agreger(len(CELLULES_T), cell_pix, idx_centre, series_t, seuils_extremes(series_t["jour"], pix_lat, "earth"))
    del series_t
    print(f"Statistiques de température de surface calculées ({time.time() - t0:.0f} s)")

    # --- MERRA-2 : air et rayonnement (grille 0,5° x 0,625°)
    import xarray as xr
    def lire_merra(nom_court, variables):
        tableaux = {v: [] for v in variables}
        for g in telecharger(nom_court):
            chemin = recuperer(g, f"{DOSSIER}/merra2")[0]
            with xr.open_dataset(chemin) as ds:
                lat_m, lon_m = ds["lat"].values, ds["lon"].values
                for v in variables:
                    tableaux[v].append(ds[v].values.reshape(len(lat_m), len(lon_m)).astype(np.float32))
            os.remove(chemin)
        return {v: np.stack(a, axis=-1) for v, a in tableaux.items()}, lat_m, lon_m   # (lat, lon, mois)

    air_m, lat_m, lon_m = lire_merra("M2SMNXSLV", ["T2MMAX", "T2MMEAN", "T2MMIN"])
    rad_m, _, _ = lire_merra("M2TMNXRAD", ["SWGDN", "LWGAB", "CLDTOT"])
    gl_lat, gl_lng = np.repeat(lat_m, len(lon_m)), np.tile(lon_m, len(lat_m))
    index_t = pd.Index(CELLULES_T)
    cp_m = index_t.get_indexer(vers_cellules(gl_lat, gl_lng))
    ic_m = (indices_grille(LAT_T, lat_m[0], lat_m[1] - lat_m[0], len(lat_m)) * len(lon_m)
            + indices_grille(LNG_T, lon_m[0], lon_m[1] - lon_m[0], len(lon_m), periodique=True))
    aplatir = lambda a: a.reshape(-1, a.shape[-1])
    air_t = agreger(len(CELLULES_T), cp_m, ic_m, {"moy": aplatir(air_m["T2MMEAN"]), "min": aplatir(air_m["T2MMIN"]), "max": aplatir(air_m["T2MMAX"])})
    rad_t = agreger(len(CELLULES_T), cp_m, ic_m, {"sw": aplatir(rad_m["SWGDN"]), "lw": aplatir(rad_m["LWGAB"]), "cld": aplatir(rad_m["CLDTOT"])})

    fraction, f_min, f_max = fractions_jour(LAT_T, CORPS["earth"]["obliquite"])
    duree = fraction * CORPS["earth"]["cycle_h"]
    nuages = rad_t["cld"]["mean"]
    illum = {"shortwave": rad_t["sw"]["mean"], "longwave": rad_t["lw"]["mean"], "cloud": nuages,
             "daylight_fraction": fraction, "daylight_min": f_min, "daylight_max": f_max,
             "daylight_h": duree, "sunshine": duree * (1 - np.nan_to_num(nuages, nan=0)),
             "n_total": rad_t["sw"]["n_total"], "n_valides": rad_t["sw"]["n_valides"]}
    ecrire("earth", construire_tables("earth", CELLULES_T, lst_t, air_t, illum))
    display(controle("earth", {"Death Valley": (36.46, -116.87), "Sahara (Hoggar)": (23.3, 5.6), "Atacama": (-24.5, -69.25),
                               "Vallées sèches (Antarctique)": (-77.5, 162.0), "Montréal": (45.5, -73.57), "Devon Island": (75.4, -89.8)}))

## Mars

**Mars Climate Database v6.2** (LMD, CNES, ESA), using the `clim_aveEUV` climatological scenario.

The database is available upon request from LMD through the [official request form](https://www-mars.lmd.jussieu.fr/MCD_pro/mcd_pro.html) and **must not be redistributed**.

Copy the 12 files, from `clim_01_me.nc` to `clim_12_me.nc`, located in the `data/clim_aveEUV` directory, into `inputs/mcd/` (subdirectories are supported).

Each file contains monthly averaged data, where one Martian "month" corresponds to **30° of solar longitude**.

### Data Dimensions

- **`Time`:** 12 time steps, sampled every 2 hours, expressed in true solar time at the prime meridian (0° longitude).
- **`altitude`:** Atmospheric altitude levels.
- **`latitude`:** 49 grid points, ranging from 90° to −90°.
- **`longitude`:** 64 grid points, ranging from −180° to 174.375°.

### Local Solar Time and Interpolation

The local solar time at a given location is calculated as:

```text
local_solar_time = prime_meridian_time + longitude / 15
```

Temperatures corresponding to MODIS observation times are obtained through interpolation between the two nearest time steps.

### Variables Used

| Variable | Description |
|---|---|
| `tsurf` | Surface temperature |
| `fluxsurf_dn_sw` | Downward shortwave (solar) radiation at the surface |
| `fluxsurf_lw` | Downward longwave (infrared) radiation at the surface |
| `temp` | Air temperature at the lowest atmospheric level |

In [ ]:
if FAIRE["mars"]:
    # Moyennes mensuelles (…_me.nc), sans les fichiers de haute atmosphère (thermo)
    fichiers_mcd = sorted(f for f in glob.glob(f"{DOSSIER_MCD}/**/*_me.nc", recursive=True)
                          if "thermo" not in os.path.basename(f).lower() and re.search(r"(\d{2})_?me\.nc$", f))
    print(f"{len(fichiers_mcd)} fichiers MCD trouvés :", [os.path.basename(f) for f in fichiers_mcd][:12])
    if not fichiers_mcd:
        print("Aucun fichier : Mars est ignoré pour cette exécution.")
        FAIRE["mars"] = False

In [ ]:
if FAIRE["mars"]:
    import xarray as xr

    VARIABLES = {"tsurf": "tsurf", "sw": "fluxsurf_dn_sw", "lw": "fluxsurf_lw", "air": "temp"}
    donnees = {k: [] for k in VARIABLES}
    for f in fichiers_mcd:
        with xr.open_dataset(f, decode_times=False) as ds:
            for cle, nom in VARIABLES.items():
                da = ds[nom]
                if "altitude" in da.dims:
                    da = da.isel(altitude=0)                                  # niveau le plus bas
                donnees[cle].append(da.transpose("Time", "latitude", "longitude").values.astype(np.float32))
            if f == fichiers_mcd[0]:
                heures_ut = ds["Time"].values.astype(float)
                lat_mcd, lon_mcd = ds["latitude"].values.astype(float), ds["longitude"].values.astype(float)
    donnees = {k: np.stack(v) for k, v in donnees.items()}   # (mois, heure, lat, lon)
    print(f"MCD : {len(fichiers_mcd)} mois, heures {heures_ut[0]:g} à {heures_ut[-1]:g}, grille {len(lat_mcd)} × {len(lon_mcd)}")

In [ ]:
if FAIRE["mars"]:
    def a_l_heure_locale(cube, heure_locale):
        # cube (mois, heure UT, lat, lon) -> (mois, lat, lon) à l'heure locale demandée, interpolation linéaire périodique.
        ordre = np.argsort(heures_ut % 24)
        h = (heures_ut % 24)[ordre]
        c = cube[:, ordre]
        h_ext = np.concatenate([h, [h[0] + 24]])
        c_ext = np.concatenate([c, c[:, :1]], axis=1)
        sortie = np.empty((cube.shape[0], cube.shape[2], cube.shape[3]), np.float32)
        for j, lon in enumerate(lon_mcd):
            ut = (heure_locale - lon / 15.0) % 24
            if ut < h[0]: ut += 24
            k = np.searchsorted(h_ext, ut) - 1
            k = min(max(k, 0), len(h) - 1)
            w = (ut - h_ext[k]) / (h_ext[k + 1] - h_ext[k])
            sortie[:, :, j] = (1 - w) * c_ext[:, k, :, j] + w * c_ext[:, k + 1, :, j]
        return sortie

    def observations(cube, heures):
        # (mois, heure, lat, lon) -> (lat*lon, mois*len(heures)) aux heures locales demandées
        obs = np.stack([a_l_heure_locale(cube, h) for h in heures], axis=-1)       # mois, lat, lon, heures
        return obs.transpose(1, 2, 0, 3).reshape(len(lat_mcd) * len(lon_mcd), -1)

    ts = donnees["tsurf"]
    series_m = {"jour": observations(ts, HEURES["jour"]), "nuit": observations(ts, HEURES["nuit"])}
    gl_lat, gl_lng = np.repeat(lat_mcd, len(lon_mcd)), np.tile(lon_mcd, len(lat_mcd))
    index = pd.Index(CELLULES)
    cp = index.get_indexer(vers_cellules(gl_lat, gl_lng))
    ic = (indices_grille(LAT_C, lat_mcd[0], lat_mcd[1] - lat_mcd[0], len(lat_mcd)) * len(lon_mcd)
          + indices_grille(LNG_C, lon_mcd[0], lon_mcd[1] - lon_mcd[0], len(lon_mcd), periodique=True))
    lst_m = agreger(len(CELLULES), cp, ic, series_m, seuils_extremes(series_m["jour"], gl_lat, "mars"))

    tout = lambda cube: cube.transpose(2, 3, 0, 1).reshape(len(lat_mcd) * len(lon_mcd), -1)   # toutes heures x mois
    a = tout(donnees["air"])
    air_m = agreger(len(CELLULES), cp, ic, {"moy": a, "min": a, "max": a})
    rad = agreger(len(CELLULES), cp, ic, {"sw": tout(donnees["sw"]), "lw": tout(donnees["lw"])})

    fraction, f_min, f_max = fractions_jour(LAT_C, CORPS["mars"]["obliquite"])
    duree = fraction * CORPS["mars"]["cycle_h"]
    n = len(CELLULES)
    illum = {"shortwave": rad["sw"]["mean"], "longwave": rad["lw"]["mean"], "cloud": np.full(n, np.nan),   # MCD : pas de nébulosité
             "daylight_fraction": fraction, "daylight_min": f_min, "daylight_max": f_max,
             "daylight_h": duree, "sunshine": duree,
             "n_total": rad["sw"]["n_total"], "n_valides": rad["sw"]["n_valides"]}
    ecrire("mars", construire_tables("mars", CELLULES, lst_m, air_m, illum))
    display(controle("mars", {"Jezero": (18.44, 77.45), "Gale": (-4.59, 137.44), "Meridiani": (-1.95, -5.53),
                              "Hellas": (-42.4, 70.5), "Olympus Mons": (18.65, -133.8), "Pôle sud": (-85, 0)}))

---

## Quick Comparison

Same columns across all three celestial bodies: a consistency check before publishing the results (`python scripts/releases.py publier thermal_light_extrems`).

In [ ]:
lignes = []
for corps, t in RESULTATS.items():
    th, il, ex = t["thermal"], t["illumination"], t["extremes"]
    lignes.append({"corps": corps, "cellules": len(th), "couverture": th.coverage.mean(),
                   "jour médian (K)": th.lst_day_mean.median(), "nuit médiane (K)": th.lst_night_mean.median(),
                   "air médian (K)": th.air_temperature_mean.median(), "solaire médian (W/m²)": il.shortwave_radiation_mean.median(),
                   "jour le plus court médian": il.daylight_fraction_min.median(), "hot moyen": ex.hot_fraction.mean(),
                   "cold moyen": ex.cold_fraction.mean(), "fiabilité moyenne": th.reliability.mean()})
display(pd.DataFrame(lignes).round(3))
for f in sorted(os.listdir(SORTIE)):
    print(f"  {f:40s} {os.path.getsize(os.path.join(SORTIE, f)) / 1e6:7.1f} Mo")